# LAB | Feature Engineering

Solución en castellano siguiendo el flujo visto en clase: inspección de datos, limpieza, transformación de variables, creación de dummies, matriz de correlación, test de asociación, train/test split, KNN y evaluación del modelo.

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [ ]:
# Librerías
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from scipy.stats import chi2_contingency

In [ ]:
# Cargar el dataset
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

**Check the shape of your data**

In [ ]:
# Comprobar filas y columnas
print("Filas y columnas:", spaceship.shape)

**Check for data types**

In [ ]:
# Comprobar tipos de datos
spaceship.dtypes

**Check for missing values**

In [ ]:
# Comprobar valores nulos por columna
spaceship.isna().sum()

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [ ]:
# Como el porcentaje de nulos es bajo, eliminamos las filas con valores nulos
spaceship_clean = spaceship.dropna().copy()

print("Shape original:", spaceship.shape)
print("Shape tras dropna:", spaceship_clean.shape)
print("Nulos restantes:", spaceship_clean.isna().sum().sum())

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [ ]:
# Cabin es demasiado granular. Nos quedamos solo con la primera letra, que representa el deck.
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str[0]

# Comprobamos las categorías resultantes
spaceship_clean["Cabin"].value_counts().sort_index()

- Drop PassengerId and Name

In [ ]:
# PassengerId y Name son identificadores/texto muy específicos, no aportan información generalizable al modelo
spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])
spaceship_clean.head()

- For non-numerical columns, do dummies.

In [ ]:
# Separar variable objetivo y variables predictoras
y = spaceship_clean["Transported"].astype(int)
X = spaceship_clean.drop(columns=["Transported"])

# Convertir variables categóricas en dummies
X = pd.get_dummies(X, drop_first=True)

# Convertimos booleanos/dummies a formato numérico para el modelo
X = X.astype(float)

print("Columnas finales de X:")
print(X.columns.tolist())

X.head()

**Matriz de correlación**

Como parte del feature engineering, revisamos la relación entre variables numéricas y la variable objetivo `Transported`. Esto ayuda a detectar qué variables parecen estar más relacionadas con el resultado.

In [ ]:
# Matriz de correlación incluyendo la variable objetivo
model_data_corr = X.copy()
model_data_corr["Transported"] = y.values

corr_matrix = model_data_corr.corr(numeric_only=True)
corr_matrix.head()

In [ ]:
# Correlación de cada variable con Transported
corr_with_target = corr_matrix["Transported"].sort_values(ascending=False)
corr_with_target

**Test de asociación para variables categóricas**

Antes de crear dummies, las variables categóricas se pueden comparar con `Transported` usando tablas de contingencia y Chi-cuadrado. Esto permite comprobar si existe asociación estadística entre una variable categórica y la variable objetivo.

In [ ]:
# Test Chi-cuadrado entre variables categóricas y Transported
categorical_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]

association_results = []

for col in categorical_cols:
    contingency_table = pd.crosstab(spaceship_clean[col], spaceship_clean["Transported"])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    association_results.append({
        "variable": col,
        "chi2": chi2,
        "p_value": p_value,
        "significant_0.05": p_value < 0.05
    })

association_df = pd.DataFrame(association_results).sort_values("p_value")
association_df

**Interpretación breve**

- La matriz de correlación ayuda a identificar variables numéricas con mayor relación lineal con `Transported`.
- El test Chi-cuadrado ayuda a detectar si las variables categóricas están asociadas con la variable objetivo.
- Estas revisiones forman parte del feature engineering porque permiten entender mejor qué información puede aportar cada variable al modelo.

**Perform Train Test Split**

In [ ]:
# División entre entrenamiento y test
# Usamos 80% para entrenar y 20% para evaluar.
# stratify=y mantiene una proporción similar de clases en train y test.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

print("Dimensión de X_train:", X_train.shape)
print("Dimensión de X_test:", X_test.shape)
print("Dimensión de y_train:", y_train.shape)
print("Dimensión de y_test:", y_test.shape)

**Escalado de variables**

KNN calcula distancias entre observaciones, por lo que las variables con escalas grandes pueden dominar el modelo. Por eso escalamos las variables después del train/test split, ajustando el scaler solo con `X_train` para evitar data leakage.

In [ ]:
# Escalar variables para KNN
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [ ]:
# Inicializar y entrenar el modelo KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train);

- Evaluate your model's performance. Comment it

In [ ]:
# Evaluar el rendimiento del modelo
train_score = knn.score(X_train_scaled, y_train)
test_score = knn.score(X_test_scaled, y_test)

print("Train score:", train_score)
print("Test score:", test_score)

**Prueba de diferentes valores de k**

Probamos varios valores de `n_neighbors` para ver cómo cambia el accuracy en test. Esto ayuda a entender que el rendimiento de KNN depende del número de vecinos elegido.

In [ ]:
# Probar diferentes valores de k
results = []

for k in range(1, 16):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    results.append({"k": k, "train_score": train_acc, "test_score": test_acc})

results_df = pd.DataFrame(results)
results_df.sort_values("test_score", ascending=False)

### Conclusión final

En este lab hemos aplicado feature engineering antes de entrenar un modelo de Machine Learning:

1. Revisamos la estructura del dataset, tipos de datos y valores nulos.
2. Eliminamos filas con nulos porque el porcentaje era bajo.
3. Transformamos `Cabin` para quedarnos solo con el deck, reduciendo la granularidad.
4. Eliminamos identificadores como `PassengerId` y `Name`, que no aportan información generalizable al modelo.
5. Convertimos variables categóricas en variables dummy.
6. Revisamos relaciones mediante matriz de correlación y test Chi-cuadrado.
7. Dividimos los datos en train/test.
8. Escalamos las variables porque KNN depende de distancias.
9. Entrenamos y evaluamos un modelo KNN.

La métrica principal usada es el accuracy mediante `.score()`. El score de test es especialmente importante porque mide el rendimiento del modelo sobre datos no vistos durante el entrenamiento.